# Chapter 21 — Keep Context in the Right World

## Question

**Can perfectly correct, current, authoritative information still be wrong context?**

Falsifiable structure: if every item is valid in its own world yet the active computation names one world, does deterministic eligibility exclude the other world's items? If yes, scope is membership — not relevance, authority, freshness, or readability.

## Setup — two similar worlds, one active computation

Projects A and B share vocabulary and shapes but conflict validly on backend, test command, and migration state. No malicious data; nothing stale; no authority violated.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Decision(Enum):
    ELIGIBLE = 'ELIGIBLE'
    REJECT_SCOPE = 'REJECT_SCOPE'

@dataclass(frozen=True)
class WorldItem:
    id: str
    text: str
    project: str
    environment: str
    task: str
    shared: bool
    tokens: int
    relevant: bool = True
    authoritative: bool = True
    fresh: bool = True
    accessible: bool = True

POOL = [
    WorldItem('a-backend', 'backend = PostgreSQL', 'A', 'main', 'migration-check', False, 60),
    WorldItem('a-test', 'test = pytest', 'A', 'main', 'migration-check', False, 40),
    WorldItem('a-mig', 'migration = complete', 'A', 'main', 'migration-check', False, 50),
    WorldItem('b-backend', 'backend = SQLite', 'B', 'main', 'any', False, 60),
    WorldItem('b-test', 'test = npm test', 'B', 'main', 'any', False, 40),
    WorldItem('b-mig', 'migration = pending', 'B', 'main', 'any', False, 50),
    WorldItem('b-bench-hypo', 'hypothesis: SQLite faster (benchmark task)', 'B', 'main', 'benchmark', False, 80),
    WorldItem('org-standard', 'organisation coding standard v4', 'shared', 'any', 'any', True, 200),
    WorldItem('b-secret', 'deploy key (readable, wrong world)', 'B', 'main', 'any', False, 30),
]
ACTIVE = {'project': 'A', 'environment': 'main', 'task': 'migration-check', 'role': 'coding-agent'}
print(f'{len(POOL)} candidates; active world: {ACTIVE}')

## Baseline — the global pool contaminates without any defect

In [ ]:
b = next(x for x in POOL if x.id == 'b-test')
print(f"Project B test item: relevant={b.relevant}, authoritative={b.authoritative}, fresh={b.fresh}, accessible={b.accessible}")
print('Every quality gate passes inside its own world — and the item still must not enter task A.')

## Intervention 1 — deterministic eligibility, three bindings deep

In [ ]:
def eligible(item, active):
    if item.shared:
        return True  # explicitly shared scope only; never a silent default
    if item.project == 'UNKNOWN' or item.environment == 'UNKNOWN':
        return False  # unknown stays unknown; never defaulted to global
    return (item.project == active['project'] and item.environment == active['environment']
            and item.task in (active['task'], 'any'))

elig = {x.id: eligible(x, ACTIVE) for x in POOL}
print('eligible:', sorted(i for i, e in elig.items() if e))
print('wrong-world:', sorted(i for i, e in elig.items() if not e))
assert elig['a-backend'] and elig['a-test'] and elig['a-mig']
assert not elig['b-backend'] and not elig['b-test'] and not elig['b-mig']
assert elig['org-standard']
required_retained = all(elig[i] for i in ('a-backend', 'a-test', 'a-mig'))
print(f'required candidates retained: {required_retained}')

## Intervention 2 — labels mark; removal un-occupies

In [ ]:
wrong = [x for x in POOL if not elig[x.id] and not x.shared]
soft_tokens = sum(x.tokens for x in wrong)  # present but tagged PROJECT B
hard_tokens = 0  # never admitted: zero occupancy, zero interference
print(f'soft separation resident wrong-world tokens: {soft_tokens}')
print(f'hard eligibility resident wrong-world tokens: {hard_tokens}')
assert soft_tokens > 0 and hard_tokens == 0
print('Tags do not remove occupancy. Only the boundary un-occupies.')

## Intervention 3 — transfer, asymmetry, access

One legitimate B dependency crosses by explicit record — origin intact, no promotion. Task hypotheses stay down unless explicitly promoted. Readability never implies eligibility, and eligibility never picks locks.

In [ ]:
transfer = {'origin_scope': 'B', 'destination_scope': 'A', 'reason': 'A benchmarks against B numbers',
            'representation': 'section', 'item': 'b-backend'}
print(f"transfer record: {transfer['item']} {transfer['origin_scope']} -> {transfer['destination_scope']} ({transfer['reason']})")
assert transfer['origin_scope'] == 'B', 'origin remains visible; transfer is not promotion'
print('transfer changes eligibility for one computation; authority and freshness untouched.')

# Asymmetry: project constraint flows down; task hypothesis does not bubble up.
print('project constraint -> task eligible: True (downward inheritance)')
print('task hypothesis -> project standing: False without explicit promotion')

secret = next(x for x in POOL if x.id == 'b-secret')
print(f"deploy key: accessible={secret.accessible}, scope_eligible={elig['b-secret']}")
assert secret.accessible is True and elig['b-secret'] is False
print('Readable is not eligible; eligible would never grant read permission.')

## Observation — every exclusion leaves a readable trace

In [ ]:
trace = [{'item_id': x.id, 'item_bindings': (x.project, x.environment, x.task),
           'active_bindings': (ACTIVE['project'], ACTIVE['environment'], ACTIVE['task']),
           'decision': 'REJECT_SCOPE', 'rule_checked': 'project+environment+task membership'}
          for x in POOL if not elig[x.id]]
for r in trace:
    print(f"{r['item_id']:13s} {r['decision']} bindings={r['item_bindings']}")
assert all(r['decision'] == 'REJECT_SCOPE' for r in trace)
print('A later miss is diagnosable as retrieval vs policy vs correct exclusion.')

## Sibling-book result (imported, not reproduced)

> SUPPORTED SIBLING-BOOK RESULT (Memory book, Ch10, frozen run `ch10-20260920T163314Z-context-frames`; 68-unit three-project corpus, fixed token budget, fixed reader): restricting candidates to the project drove cross-project leakage from 0.212 to zero with must-include recall held exactly at 0.773 and precision up; similar-fixture probe mean leakage 0.667 to zero. Scope: that corpus, reader, and eleven tasks only — consumed as controlled-corpus phenomenon, not as the sibling's ProjectFrame architecture.

The executable cells above test membership mechanics, not those numbers.

## Try it

1. Retask ACTIVE to project B and confirm the eligible set mirrors exactly.
2. Mark `org-standard` shared=False and watch it correctly excluded — shared must be explicit.
3. Admit `b-bench-hypo` into the release task and state which binding refused it and why.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(eligible(next(x for x in POOL if x.id == 'b-backend'), ACTIVE))

## What this demonstrates

- Scope is eligibility to influence one computation — not readability, relevance, freshness, or authority.
- Correct information from the wrong world is wrong context for this computation.
- Explicit transfer and explicit shared scope differ from silently global information.

## What this does not demonstrate

- That scope enforcement improves model behaviour, or that project-root filtering suffices universally.
- That labels are useless, or that cross-project sharing should never happen.
- That scope policy authenticates, encrypts, or enforces privacy.
- That transfer grants authority, or that eligibility means admission.

## Connection to the chapter

Every constraint is now on the table at once:

> We now have relevance, retention, authority, freshness, scope, legal representations, dependencies, and a finite budget. They must become one concrete bundle—or an explicit failure.

That is Chapter 22.